# 02 Cleaning

**Cleaning steps covered:**
1. Load raw data via the ETL pipeline
2. Validate numeric ranges
3. Handle missing values
4. Standardise categorical labels
5. Remove outliers using IQR fencing
6. Export cleaned dataset

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import basic_clean

RAW_PATH       = PROJECT_ROOT / 'data/raw/student_mental_health_burnout_1M.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

print(f'Raw path      : {RAW_PATH}')
print(f'Processed path: {PROCESSED_PATH}')

## 2.1 Load and Apply Basic Clean

In [ ]:
raw_df = pd.read_csv(RAW_PATH)
df = basic_clean(raw_df)   # normalise column names, drop duplicates, strip whitespace

print(f'Raw rows   : {len(raw_df):,}')
print(f'After basic_clean: {len(df):,} rows, {len(df.columns)} columns')
df.head()

## 2.2 Validate Numeric Ranges

Expected ranges based on domain knowledge:

| Column | Min | Max |
|---|---|---|
| age | 17 | 30 |
| academic_year | 1 | 4 |
| study_hours_per_day | 0 | 24 |
| exam_pressure | 0 | 10 |
| academic_performance | 0 | 100 |
| stress_level | 0 | 10 |
| anxiety_score | 0 | 10 |
| depression_score | 0 | 10 |
| sleep_hours | 0 | 12 |
| physical_activity | 0 | 10 |
| social_support | 0 | 10 |
| screen_time | 0 | 16 |
| internet_usage | 0 | 16 |
| financial_stress | 0 | 10 |
| family_expectation | 0 | 10 |
| burnout_score | 0 | 10 |
| mental_health_index | 0 | 10 |
| dropout_risk | 0 | 10 |

In [ ]:
RANGE_RULES = {
    'age':                   (17, 30),
    'academic_year':         (1, 4),
    'study_hours_per_day':   (0, 24),
    'exam_pressure':         (0, 10),
    'academic_performance':  (0, 100),
    'stress_level':          (0, 10),
    'anxiety_score':         (0, 10),
    'depression_score':      (0, 10),
    'sleep_hours':           (0, 12),
    'physical_activity':     (0, 10),
    'social_support':        (0, 10),
    'screen_time':           (0, 16),
    'internet_usage':        (0, 16),
    'financial_stress':      (0, 10),
    'family_expectation':    (0, 10),
    'burnout_score':         (0, 10),
    'mental_health_index':   (0, 10),
    'dropout_risk':          (0, 10),
}

violations = {}
for col, (lo, hi) in RANGE_RULES.items():
    if col in df.columns:
        mask = (df[col] < lo) | (df[col] > hi)
        count = mask.sum()
        if count > 0:
            violations[col] = count

if violations:
    print('Range violations found:')
    for col, cnt in violations.items():
        print(f'  {col}: {cnt:,} rows out of range')
else:
    print('All numeric columns are within expected ranges.')

## 2.3 Handle Missing Values

In [ ]:
missing_before = df.isnull().sum()
print('Missing values before imputation:')
print(missing_before[missing_before > 0])

# Numeric columns: fill with column median (robust to skew)
num_cols = df.select_dtypes(include='number').columns
for col in num_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  Filled {col} nulls with median={median_val:.4f}')

# Categorical columns: fill with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'  Filled {col} nulls with mode={mode_val}')

print(f'\nMissing values after imputation: {df.isnull().sum().sum()}')

## 2.4 Standardise Categorical Labels

In [ ]:
# Standardise gender to Title Case (Male / Female / Other)
df['gender'] = df['gender'].str.strip().str.title()
print('gender values:', df['gender'].unique())

# Standardise risk_level to Title Case (Low / Medium / High)
df['risk_level'] = df['risk_level'].str.strip().str.title()
print('risk_level values:', df['risk_level'].unique())

## 2.5 Clip Extreme Values to Valid Ranges

Any values outside the defined domain ranges are clipped rather than dropped, preserving row count.

In [ ]:
rows_before = len(df)

for col, (lo, hi) in RANGE_RULES.items():
    if col in df.columns:
        df[col] = df[col].clip(lower=lo, upper=hi)

print(f'Rows before: {rows_before:,} | Rows after: {len(df):,}')
print('All values clipped to valid domain ranges.')

## 2.6 Data Type Enforcement

In [ ]:
# academic_year should be integer
df['academic_year'] = df['academic_year'].round().astype(int)

# age should be integer
df['age'] = df['age'].round().astype(int)

# gender and risk_level as category
df['gender']     = df['gender'].astype('category')
df['risk_level'] = df['risk_level'].astype('category')

print('Data types after enforcement:')
print(df.dtypes)

## 2.7 Final Quality Check

In [ ]:
print(f'Final shape: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')
print(f'Remaining duplicates: {df.duplicated().sum():,}')
df.describe(include='all').T

## 2.8 Export Cleaned Dataset

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to {PROCESSED_PATH}')
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')

## 2.9 Cleaning Summary

| Step | Action | Outcome |
|---|---|---|
| basic_clean | Normalise column names, drop duplicates, strip whitespace | Column names in snake_case |
| Range validation | Check all numeric columns against domain bounds | Violations flagged |
| Missing values | Median imputation for numeric, mode for categorical | Zero nulls remaining |
| Categorical labels | Title-case standardisation for gender and risk_level | Consistent labels |
| Clipping | Clip values to valid domain ranges | No out-of-range values |
| Data types | age and academic_year cast to int; categoricals typed | Correct dtypes |

